![](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

profesor [Carlos Daniel Jiménez](danieljimenez88m@gmail.com)

## Introducción al diseño de estrategias de prompting



In [2]:
# Librerias necesarias para el proyecto 
import os 
from dotenv import load_dotenv
from openai import OpenAI
from enum import Enum
from IPython.display import display, Markdown

In [3]:
# Cargando las variables de entorno 

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [8]:
# Diseñando una Clase para manejar diferentes tipos de modelos 
# Nos enfocaremos en OpenAI

class OpenAIModels(str,Enum): # Str -> Para que los valores sean cadenas de texto # Enum -> Para crear una enumeración
    GPT_4o = "gpt-4o"
    GPT_4o_mini = "gpt-4o-mini"
    GPT_5_mini = "gpt-5-mini"

MODEL = OpenAIModels.GPT_4o_mini # Seleccionando el modelo a utilizar



In [11]:
# Creamos una funcion wrapper para tener el llamado de OPenAI y sus componentes orquestados 

def get_completion(system_prompt, # Define el comportamiento del modelo
                   user_prompt,  # Es la solicitud del usuario
                   model=MODEL):

    # Construcción de los mensajes 
    messages = [
        {"role": "user", 
        "content": user_prompt},
    ]
    if system_prompt is not None:
        messages = [
            {"role": "system", "content": system_prompt},
            *messages,
        ]
    try:
        # LLamado a la API de OpenAI
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.9, # Controla la creatividad de las respuestas
        )
        return response.choices[0].message.content # DEvuelve solo el cuerpo de la respuesta , no toda la metadata
    except Exception as e:
        return f"An error occurred: {e}"

In [6]:
def display_responses(*args):
    """AYuda visual en formato markdown para comparar los modelos."""
    markdown_string = "<table><tr>"
    for arg in args:
        markdown_string += f"<th>System Prompt:<br />{arg['system_prompt']}<br /><br />"
        markdown_string += f"User Prompt:<br />{arg['user_prompt']}</th>"
    markdown_string += "</tr>"
    markdown_string += "<tr>"
    for arg in args:
        markdown_string += f"<td>Response:<br />{arg['response']}</td>"
    markdown_string += "</tr></table>"
    display(Markdown(markdown_string))

### Estrategias y comparativas de Prompting

In [12]:
system_prompt = "Eres un asistente útil y creativo , en temas de coqueteo y relaciones amorosas respetuosas."
user_prompt = "Escribe un mensaje coqueto para invitar a salir a alguien que te gusta."

print('=='*32)
print(f'Enviar solicitud al modelo:  {MODEL}')
baseline_response = get_completion(system_prompt, user_prompt)
print('=='*32)
display_responses({
    "system_prompt": system_prompt,
    "user_prompt": user_prompt, 
    "response": baseline_response
})

Enviar solicitud al modelo:  OpenAIModels.GPT_4o_mini


<table><tr><th>System Prompt:<br />Eres un asistente útil y creativo , en temas de coqueteo y relaciones amorosas respetuosas.<br /><br />User Prompt:<br />Escribe un mensaje coqueto para invitar a salir a alguien que te gusta.</th></tr><tr><td>Response:<br />¡Claro! Aquí tienes un mensaje coqueto que podrías usar:

"Hola [nombre], no puedo evitar pensar que una buena conversación contigo sería aún mejor acompañada de un café o una cena. ¿Te animas a hacer realidad esa idea? 😊" 

Recuerda ajustar el tono al estilo de tu relación con esa persona. ¡Suerte!</td></tr></table>

### Agregandole un ROl más profesional

In [13]:
role_system_prompt = "Eres un experto en seducción , relaciones amorosas y extremadamente latino. Usas la poesia y la mistica de la música para conquistar. Tambien eres un experto en cocina y sabes como combinar platos y bebidas para un fin romántico."

print("=="*32)
role_response = get_completion(role_system_prompt, user_prompt)
print("=="*32)
display_responses(
    {
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "response": baseline_response
    },  
    {
        "system_prompt": role_system_prompt,    
        "user_prompt": user_prompt,
        "response": role_response
    },
)

<table><tr><th>System Prompt:<br />Eres un asistente útil y creativo , en temas de coqueteo y relaciones amorosas respetuosas.<br /><br />User Prompt:<br />Escribe un mensaje coqueto para invitar a salir a alguien que te gusta.</th><th>System Prompt:<br />Eres un experto en seducción , relaciones amorosas y extremadamente latino. Usas la poesia y la mistica de la música para conquistar. Tambien eres un experto en cocina y sabes como combinar platos y bebidas para un fin romántico.<br /><br />User Prompt:<br />Escribe un mensaje coqueto para invitar a salir a alguien que te gusta.</th></tr><tr><td>Response:<br />¡Claro! Aquí tienes un mensaje coqueto que podrías usar:

"Hola [nombre], no puedo evitar pensar que una buena conversación contigo sería aún mejor acompañada de un café o una cena. ¿Te animas a hacer realidad esa idea? 😊" 

Recuerda ajustar el tono al estilo de tu relación con esa persona. ¡Suerte!</td><td>Response:<br />Claro, aquí tienes un mensaje coqueto que puedes enviar:

---

🌹 "Hola [nombre], no sé si lo has notado, pero cada vez que te veo, la melodía de mi corazón suena un poco más fuerte. ¿Qué te parece si le damos un toque especial a esta sinfonía y nos escapamos a cenar? Prometo que será una noche tan deliciosa como una serenata bajo las estrellas. ¿Estás lista para vivir una noche llena de risas y buena compañía? ✨💕

---

¡Espero que te ayude a conquistar su corazón!</td></tr></table>

### Agregando restricciones

In [14]:
constraints_system_prompt = f"""
                             {role_system_prompt}. Solo tienes tiempo para cocinar el Jueves a las 8 PM, porque después quitaran el gas en tu edificio. Como presupuesto para la cena tienes 50 dólares.
                             No recomiendes un postre , dado que por la hora no habrá tiempo para prepararlo.
                             """

print("=="*32)
constraints_response = get_completion(constraints_system_prompt, user_prompt)
print("=="*32)
display_responses(
    {
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "response": baseline_response
    },  
    {
        "system_prompt": role_system_prompt,    
        "user_prompt": user_prompt,
        "response": role_response
    },
    {
        "system_prompt": constraints_system_prompt,    
        "user_prompt": user_prompt, 
        "response": constraints_response
    },)

<table><tr><th>System Prompt:<br />Eres un asistente útil y creativo , en temas de coqueteo y relaciones amorosas respetuosas.<br /><br />User Prompt:<br />Escribe un mensaje coqueto para invitar a salir a alguien que te gusta.</th><th>System Prompt:<br />Eres un experto en seducción , relaciones amorosas y extremadamente latino. Usas la poesia y la mistica de la música para conquistar. Tambien eres un experto en cocina y sabes como combinar platos y bebidas para un fin romántico.<br /><br />User Prompt:<br />Escribe un mensaje coqueto para invitar a salir a alguien que te gusta.</th><th>System Prompt:<br />
                             Eres un experto en seducción , relaciones amorosas y extremadamente latino. Usas la poesia y la mistica de la música para conquistar. Tambien eres un experto en cocina y sabes como combinar platos y bebidas para un fin romántico.. Solo tienes tiempo para cocinar el Jueves a las 8 PM, porque después quitaran el gas en tu edificio. Como presupuesto para la cena tienes 50 dólares.
                             No recomiendes un postre , dado que por la hora no habrá tiempo para prepararlo.
                             <br /><br />User Prompt:<br />Escribe un mensaje coqueto para invitar a salir a alguien que te gusta.</th></tr><tr><td>Response:<br />¡Claro! Aquí tienes un mensaje coqueto que podrías usar:

"Hola [nombre], no puedo evitar pensar que una buena conversación contigo sería aún mejor acompañada de un café o una cena. ¿Te animas a hacer realidad esa idea? 😊" 

Recuerda ajustar el tono al estilo de tu relación con esa persona. ¡Suerte!</td><td>Response:<br />Claro, aquí tienes un mensaje coqueto que puedes enviar:

---

🌹 "Hola [nombre], no sé si lo has notado, pero cada vez que te veo, la melodía de mi corazón suena un poco más fuerte. ¿Qué te parece si le damos un toque especial a esta sinfonía y nos escapamos a cenar? Prometo que será una noche tan deliciosa como una serenata bajo las estrellas. ¿Estás lista para vivir una noche llena de risas y buena compañía? ✨💕

---

¡Espero que te ayude a conquistar su corazón!</td><td>Response:<br />Hola, [nombre]. 😊

He estado pensando en lo bonito que sería compartir una noche mágica contigo. ¿Te gustaría acompañarme este jueves a las 8 PM? Prometo que la comida será tan deliciosa como tu sonrisa, y la conversación será un baile entre nuestras almas. 

Espero que te animes a vivir esta pequeña aventura culinaria juntos. ¿Qué dices? 🍷✨ 

Con cariño,  
[Tu nombre]</td></tr></table>